# 04 — RG-improved Schwarzschild geometry

Bonanno–Reuter (2000): promote Newton's constant to G(r) by
evaluating an RG trajectory at scale k(r). The improved metric has
a regular core (no singularity) and a critical mass below which no
horizon forms.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from asymsafety.beta.einstein_hilbert import build_eh_beta_system
from asymsafety.analysis.fixed_points import FixedPointFinder
from asymsafety.analysis.flow import FlowIntegrator
from asymsafety.cosmology.scale_identification import InverseDistanceScale
from asymsafety.cosmology.rg_improved_bh import RGImprovedSchwarzschild


## 1. Build a UV→IR trajectory

Start from just inside the basin of the NGFP and flow toward small t (IR).


In [ ]:
system = build_eh_beta_system(d=4)
ngfp = FixedPointFinder(system).find_fixed_point({'g': 0.7, 'lambda': 0.14})
ic_uv = {'g': ngfp.location['g'] - 0.001, 'lambda': ngfp.location['lambda'] + 0.001}
traj = FlowIntegrator(system).integrate(ic_uv, t_span=(10, -10), max_step=0.05)
print(f'Trajectory has {len(traj.t_values)} points.')


## 2. RG-improved geometry for an O(1) mass


In [ ]:
bh = RGImprovedSchwarzschild(traj, scale=InverseDistanceScale(xi=1.0), M=1.0, k0=1.0)
rs = np.geomspace(1e-3, 100, 400)
G_r = np.array([bh.G(r) for r in rs])
f_r = np.array([bh.lapse(r) for r in rs])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].loglog(rs, G_r); ax[0].set_xlabel('r'); ax[0].set_ylabel('G(r)')
ax[0].set_title('Running Newton constant — G(r→0) → 0')
ax[0].grid(True, which='both', alpha=0.3)
ax[1].semilogx(rs, f_r); ax[1].axhline(0, color='k', lw=0.5)
ax[1].set_xlabel('r'); ax[1].set_ylabel('f(r)'); ax[1].set_title('Lapse — f(r→0) → 1 (de Sitter core)')
ax[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 3. Critical mass — no horizon below it


In [ ]:
M_crit = bh.critical_mass(M_search=(1e-4, 5.0))
print(f'Critical mass M_crit ≈ {M_crit:.4e} (in units k0=1)')

for M in [1e-4, M_crit*0.5, M_crit*2.0, 1.0, 10.0]:
    bhM = RGImprovedSchwarzschild(traj, M=M, k0=1.0)
    h = bhM.horizons(1e-3, 1000, 5000)
    label = f'M = {M:.3e}'
    print(f'  {label:25s}: {len(h)} horizon(s) at {h}')


**Key takeaways:**

- `G(r)` falls to zero at the origin, regularizing the geometry.
- The lapse `f(r)` returns to 1 at the origin — no curvature singularity.
- For `M < M_crit`, the gravitational well is too shallow to form a horizon.
